# A.5: BPE-dropout

Encode a word several times while skipping each candidate merge with probability `p`.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 Character-level BPE training
from collections import Counter

def merge_word(word, pair):
    """Replace each occurrence of `pair` in the tuple `word` with the merged symbol."""
    out, i = [], 0
    while i < len(word):
        if i + 1 < len(word) and (word[i], word[i + 1]) == pair:
            out.append(word[i] + word[i + 1])
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)

def train_bpe(word_freqs, num_merges):
    words = {tuple(w): n for w, n in word_freqs.items()}
    merges = []
    for step in range(num_merges):
        stats = Counter()
        for w, n in words.items():
            for pair in zip(w, w[1:]):
                stats[pair] += n
        pair = max(stats, key=stats.get)          # ties: the pair seen first
        merges.append(pair)
        words = {merge_word(w, pair): n for w, n in words.items()}
        print(step + 1, pair, stats[pair], list(words))
    return merges

# From A.1 Character-level BPE training
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

merges = train_bpe(corpus, 6)


In [ ]:
import random

def encode_word_dropout(word, merges, p, rng):
    ranks = {pair: r for r, pair in enumerate(merges)}
    symbols = tuple(word)
    while len(symbols) >= 2:
        candidates = [pr for pr in zip(symbols, symbols[1:])
                      if pr in ranks and rng.random() >= p]
        if not candidates:
            break
        best = min(candidates, key=ranks.get)
        symbols = merge_word(symbols, best)
    return list(symbols)

rng = random.Random(0)
print([encode_word_dropout("lowest", merges, 0.5, rng) for _ in range(4)])


**Expected output**

Output:

```text
[['l', 'o', 'w', 'es', 't'], ['low', 'e', 's', 't'], ['lo', 'w', 'est'], ['l', 'o', 'w', 'est']]
```
